# Predicting the value of monitoring an event

Compare occurrence classifiers, direct dollar regression and two-stage models using identical eligible events, chronological folds and execution-derived targets. The primary decision is which five events to monitor. Actual strategy P&L is evaluated in notebook 03; selected standalone label sums here are diagnostics.

In [1]:
from pathlib import Path
import os
import sys
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/config.py').exists())
sys.path.insert(0, str(ROOT))
import json
import pandas as pd
from IPython.display import display
pd.set_option('display.max_columns', 20)
from src.reporting.workflow import profitability_dataset, profitability_run, execution_path
from src.models.profitability import fit_profitability
from src.reporting.profitability import PRIMARY
from src.models.settings import method_name
from src.reporting.workflow import new_profitability_run, select_profitability_run


## Models and fitting controls

Classifiers predict at least one positive attempt. Direct regressors predict signed hourly net P&L. Two-stage models multiply an unweighted classifier's probability of any attempted trade by a regressor's conditional net P&L. Conditional training includes losing and zero-return trading hours; there is no hard gate threshold. Linear, random forest and boosted-tree variants are compared.

Classifier ensembles average within-scan percentile ranks. Economic ensembles average predicted dollars. All models use book-only or book-plus-metadata inputs. Training expands through three time folds, purging the full target horizon plus 1.1 seconds. Preprocessing sees training rows only. `RETRAIN = True` saves a fresh completed run; the default inspects saved outputs.

In [2]:
RETRAIN = False
dataset_dir = profitability_dataset()
if RETRAIN:
    destination = new_profitability_run()
    fit_profitability(destination, dataset_dir)
    select_profitability_run(destination)
run_dir = profitability_run()
design = json.loads((run_dir / 'design.json').read_text())
display(pd.read_csv(run_dir / 'diagnostics/fit_audit.csv').query("model == 'reg_boosted_trees'"))
metrics = pd.read_csv(run_dir / 'metrics.csv')
main = metrics.query('top_n == 5').set_index('model')
names = PRIMARY + [method_name(name, 'combined') for name in PRIMARY] + ['quoted_gap_rule','depth_rule','freshness_rule']
display(main.reindex(names)[['kind','recall','precision_known','average_precision','mae_usd','rmse_usd',
    'selected_standalone_pnl_usd','selected_unknown']].round(4))
print('Always-zero predictor MAE / RMSE:', main.zero_prediction_mae_usd.iloc[0], main.zero_prediction_rmse_usd.iloc[0])


,fold,model,training_rows,trading_rows,loss_hours,train_last_time,test_first_time
5,nov_early,reg_boosted_trees,4859,69,1,2025-10-31 22:00:00+00:00,2025-11-01 00:00:00+00:00
23,nov_late,reg_boosted_trees,11191,234,7,2025-11-21 22:00:00+00:00,2025-11-22 00:00:00+00:00
41,dec_late,reg_boosted_trees,17690,405,11,2025-12-14 22:00:00+00:00,2025-12-15 00:00:00+00:00


,kind,recall,precision_known,average_precision,mae_usd,rmse_usd,selected_standalone_pnl_usd,selected_unknown
model,,,,,,,,
logistic,classifier,0.9734,0.0478,0.5346,NaN,NaN,5852.9923,374
random_forest,classifier,0.9521,0.0531,0.4637,NaN,NaN,5893.7376,1297
boosted_trees,classifier,0.9441,0.0532,0.5429,NaN,NaN,5169.4075,1367
rank_ensemble,classifier,0.9654,0.0528,0.1642,NaN,NaN,5893.5749,1165
reg_ridge,regression,0.9149,0.0441,0.3207,1.1670,8.4003,5548.3121,235
reg_random_forest,regression,0.8910,0.0512,0.2004,0.5144,8.3538,5574.7536,1492
reg_boosted_trees,regression,0.6489,0.0389,0.1630,0.6133,8.4568,5313.6408,1755
reg_ensemble,regression,0.8697,0.0460,0.2079,0.7225,8.3164,5647.3658,918
two_stage_ridge,two_stage,0.8218,0.0418,0.2755,0.5576,8.1508,5831.4639,649


Always-zero predictor MAE / RMSE: 0.378558548681957 8.501180107341268


## Monitoring budget and transfer

Capture, precision and selected standalone value are shown for N = 1, 3, 5 and 10. Persistent trading replay uses the primary N = 5 budget. Held-out-family diagnostics remove the evaluated topic from training while respecting chronology. They use the boosted classifier, direct regressor and two-stage model in both feature views.

In [3]:
display(metrics.loc[metrics.model.isin(['logistic','reg_boosted_trees','two_stage_boosted_trees','quoted_gap_rule']),
    ['model','top_n','recall','precision_known','selected_standalone_pnl_usd','selected_unknown']].round(4))
display(main.loc[main.index.str.startswith('family_holdout_'),
    ['recall','mae_usd','rmse_usd','selected_standalone_pnl_usd']].round(4))


,model,top_n,recall,precision_known,selected_standalone_pnl_usd,selected_unknown
84,logistic,1,0.6170,0.1437,5023.5356,18
85,logistic,3,0.8883,0.0697,5821.9709,105
86,logistic,5,0.9734,0.0478,5852.9923,374
87,logistic,10,0.9973,0.0293,5902.0086,1971
88,quoted_gap_rule,1,0.4681,0.1612,3174.6391,540
89,quoted_gap_rule,3,0.7128,0.0749,4312.1892,1316
90,quoted_gap_rule,5,0.9176,0.0529,5939.7191,1518
91,quoted_gap_rule,10,1.0000,0.0314,5902.0226,2783
180,reg_boosted_trees,1,0.3617,0.0996,3412.1943,266
181,reg_boosted_trees,3,0.5771,0.0570,4279.2726,1087


,recall,mae_usd,rmse_usd,selected_standalone_pnl_usd
model,,,,
family_holdout_boosted_trees,0.8431,NaN,NaN,5248.8884
family_holdout_combined_boosted_trees,0.8351,NaN,NaN,5251.6131
family_holdout_combined_reg_boosted_trees,0.8059,0.5230,8.5780,5847.2092
family_holdout_combined_two_stage_boosted_trees,0.6782,0.3983,8.4814,3937.6214
family_holdout_reg_boosted_trees,0.8457,0.5177,8.5777,5829.2719
family_holdout_two_stage_boosted_trees,0.7739,0.4045,8.4791,5569.6795


## Do activity probabilities match observed frequencies?

Dollar multiplication needs probability estimates, rather than arbitrary rank scores. Gates fit the observed class prevalence without class weighting or oversampling. Their outputs are not assumed calibrated. The following out-of-sample reliability bins compare predicted probabilities with observed trading frequency. Brier scores repeat across bins of the same fold. Sparse high-probability bins provide weak evidence.

In [4]:
reliability = pd.read_csv(run_dir / 'diagnostics/gate_reliability.csv')
display(reliability.loc[reliability.model.eq('two_stage_boosted_trees')].round(5))
display(reliability[['model','fold','fold_brier']].drop_duplicates().pivot(index='model',columns='fold',values='fold_brier').round(5))


,model,fold,probability_bin,rows,mean_probability,observed_activity,fold_brier
81,two_stage_boosted_trees,dec_late,"(-0.0001, 0.001]",2664,0.00014,0.00000,0.00743
82,two_stage_boosted_trees,dec_late,"(0.001, 0.005]",468,0.00212,0.00427,0.00743
83,two_stage_boosted_trees,dec_late,"(0.005, 0.01]",158,0.00821,0.01266,0.00743
84,two_stage_boosted_trees,dec_late,"(0.01, 0.02]",447,0.01433,0.01342,0.00743
85,two_stage_boosted_trees,dec_late,"(0.02, 0.05]",259,0.02995,0.04247,0.00743
86,two_stage_boosted_trees,dec_late,"(0.05, 0.1]",84,0.07069,0.10714,0.00743
87,two_stage_boosted_trees,dec_late,"(0.1, 0.2]",19,0.12908,0.00000,0.00743
88,two_stage_boosted_trees,dec_late,"(0.2, 0.5]",9,0.31688,0.33333,0.00743
89,two_stage_boosted_trees,dec_late,"(0.5, 1.0]",16,0.98426,1.00000,0.00743
90,two_stage_boosted_trees,nov_early,"(-0.0001, 0.001]",4477,0.00031,0.00223,0.01553


fold,dec_late,nov_early,nov_late
model,,,
combined_two_stage_boosted_trees,0.00777,0.01577,0.01551
combined_two_stage_random_forest,0.00800,0.01810,0.01511
combined_two_stage_ridge,0.00858,0.01747,0.01470
two_stage_boosted_trees,0.00743,0.01553,0.01540
two_stage_random_forest,0.00789,0.01781,0.01521
two_stage_ridge,0.00870,0.01584,0.01511


## Feature findings

Permutation shuffles a feature or group between events at the same scan, then measures the change in selected standalone net-P&L labels. Positive drops indicate useful information under this diagnostic. These are not persistent replay returns or causal effects. Feature-removal models are refitted; their actual replay returns appear in notebook 03 and the report.

In [5]:
importance = pd.read_csv(run_dir / 'diagnostics/permutation_importance.csv')
grouped = importance.loc[importance.unit.str.startswith('group:')].groupby(['model','unit']).selected_standalone_pnl_drop_usd.mean()
display(grouped.unstack('model').round(3))
individual = importance.loc[importance.model.eq('two_stage_boosted_trees') & importance.unit.str.startswith('feature:')]
display(individual.groupby('unit').selected_standalone_pnl_drop_usd.mean().sort_values(ascending=False).round(3))
display(main.loc[main.index.str.contains('_without_'),['recall','mae_usd','selected_standalone_pnl_usd']].round(3))
print('Final test opened: False')


model,combined_reg_boosted_trees,combined_two_stage_boosted_trees,reg_boosted_trees,two_stage_boosted_trees
unit,,,,
group:depth_size,-44.110,43.735,-48.107,56.024
group:freshness_quality,215.955,-11.234,291.853,104.413
group:history,377.436,972.739,393.522,972.266
group:metadata,27.324,-2.559,NaN,NaN
group:quoted_gap,1086.983,707.264,953.783,421.627
group:spread,-86.127,15.496,-58.618,30.878


unit
feature:x_best_quoted_net_usd                     46.728
feature:x_ask_depth_sum                           35.186
feature:x_spread_mean                              5.009
feature:x_size_feasible_package_fraction           2.708
feature:x_active_tokens                            0.490
feature:x_active_packages                          0.105
feature:x_seen_fraction                            0.000
feature:x_positive_quoted_net_package_fraction    -0.020
feature:x_clean_fraction                          -0.075
feature:x_spread_max                              -3.022
feature:x_book_age_max_s                          -7.902
Name: selected_standalone_pnl_drop_usd, dtype: float64

,recall,mae_usd,selected_standalone_pnl_usd
model,,,
reg_without_freshness_quality,0.665,0.583,5218.866
reg_without_history,0.684,0.561,5080.524
reg_without_quoted_gap,0.606,0.804,3070.986
two_stage_without_freshness_quality,0.854,0.470,5860.565
two_stage_without_history,0.827,0.458,5790.311
two_stage_without_quoted_gap,0.809,0.591,5694.999


Final test opened: False


A small prediction error can result from predicting zero in most hours. Economic evaluation must test whether rankings allocate monitoring capacity well and retain downside. Continue with [chronological trading replay](03_simulator.ipynb).